In [1]:
import torch
import torch.nn as nn

In [7]:
from pathlib import Path

from sdrebuilt.model.unet import UNet
from sdrebuilt.convert_weights import load_unet


unet = UNet()
weights_path = Path("..") / "data" / "weights" / "v1-5-pruned-emaonly.safetensors"
load_unet(weights_path, unet=unet)

In [14]:
for name, module in unet.named_modules():
    print(name)


time_emb
time_emb.linear1
time_emb.linear2
down_blocks
down_blocks.0
down_blocks.0.0
down_blocks.1
down_blocks.1.0
down_blocks.1.0.shortcut
down_blocks.1.0.groupnorm1
down_blocks.1.0.conv1
down_blocks.1.0.time
down_blocks.1.0.groupnorm2
down_blocks.1.0.conv2
down_blocks.1.1
down_blocks.1.1.groupnorm
down_blocks.1.1.proj_in
down_blocks.1.1.layernorm1
down_blocks.1.1.q1
down_blocks.1.1.k1
down_blocks.1.1.v1
down_blocks.1.1.proj_out1
down_blocks.1.1.layernorm2
down_blocks.1.1.q2
down_blocks.1.1.k2
down_blocks.1.1.v2
down_blocks.1.1.proj_out2
down_blocks.1.1.layernorm3
down_blocks.1.1.linear_geglu1
down_blocks.1.1.linear_geglu2
down_blocks.1.1.proj_out
down_blocks.2
down_blocks.2.0
down_blocks.2.0.shortcut
down_blocks.2.0.groupnorm1
down_blocks.2.0.conv1
down_blocks.2.0.time
down_blocks.2.0.groupnorm2
down_blocks.2.0.conv2
down_blocks.2.1
down_blocks.2.1.groupnorm
down_blocks.2.1.proj_in
down_blocks.2.1.layernorm1
down_blocks.2.1.q1
down_blocks.2.1.k1
down_blocks.2.1.v1
down_blocks.2.1.pr

In [17]:
from sdrebuilt.lora.layers import LoRALinear

# prototype lora layer injection function (works for self and cross attention layers only)

targets = ["q1", "k1", "v1", "proj_out1", "q2", "k2", "v2", "proj_out2"]

to_replace = [] # (parent_name, module_name, module object)

# create list of layers to be replaced
for name, module in unet.named_modules():
    parent_name, _, module_name = name.rpartition(".") # ("down_blocks.1.1", ".", "q2")
    
    if module_name in targets and isinstance(module, nn.Linear):
        to_replace.append((parent_name, module_name, module))

# inject lora layers
for parent_name, module_name, module in to_replace:
    to_inject = LoRALinear(
        base_layer=module,
        r=16,
        alpha=16
    )
    parent = unet.get_submodule(parent_name)
    setattr(parent, module_name, to_inject)


# unet should now have old q1s, k1s, etc. replaced with LoRALinear
# wrappers, with old linear layer not tracking gradients.